# Dataset Verification — Module 00 Raw Data / 模块 00 原始数据集验证

**Purpose / 目的**

Audit every file in `00_raw_data/phage/` and `00_raw_data/bacteria/`, compute SHA-256 checksums, validate FASTA parseability, cross-check against `phage_list.csv` and `bacteria_list.csv`, and write `MANIFEST.csv` conforming to `INTERFACE.md §Universal conventions`.

审计 `00_raw_data/phage/` 和 `00_raw_data/bacteria/` 中的每个文件，计算 SHA-256 校验和，验证 FASTA 格式，与 `phage_list.csv` 和 `bacteria_list.csv` 交叉核对，并写出符合 `INTERFACE.md §Universal conventions` 的 `MANIFEST.csv`。

**References / 参考文献**

- Cock, P.J.A. et al. (2009) Biopython. *Bioinformatics* 25:1422.
- FASTA format: https://en.wikipedia.org/wiki/FASTA_format
- INTERFACE.md: `<repo_root>/INTERFACE.md`

**Convention / 编写约定**

This notebook follows the project's bilingual (English + Chinese) development convention. Once it runs end-to-end and outputs match the verification cell, freeze it as `01_verify_dataset.py`.

本 notebook 遵循项目的双语（英文 + 中文）开发约定。等它能端到端跑通并通过验证 cell 后，冻结为 `01_verify_dataset.py`。

In [ ]:
# Cell 2 — Imports, path anchoring, and library versions
# Cell 2 — 引入依赖、路径锚定、打印库版本
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from Bio import SeqIO
import Bio
import numpy

# Anchor paths to repo root — never hard-code absolute paths
# 路径锚定到 repo root，绝对不要写死绝对路径
REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks live in <module>/processes/
MODULE_ROOT = REPO_ROOT / "00_raw_data"
PHAGE_ROOT = MODULE_ROOT / "phage"
BACTERIA_ROOT = MODULE_ROOT / "bacteria"

print(f"REPO_ROOT:    {REPO_ROOT}")
print(f"MODULE_ROOT:  {MODULE_ROOT}  exists={MODULE_ROOT.exists()}")
print(f"PHAGE_ROOT:   {PHAGE_ROOT}   exists={PHAGE_ROOT.exists()}")
print(f"BACTERIA_ROOT:{BACTERIA_ROOT} exists={BACTERIA_ROOT.exists()}")
print()

# Library versions / 库版本
print(f"biopython: {Bio.__version__}")
print(f"pandas:    {pd.__version__}")
print(f"numpy:     {numpy.__version__}")

# Git commit SHA / Git commit SHA
try:
    GIT_SHA = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, text=True, stderr=subprocess.DEVNULL
    ).strip()
except Exception:
    GIT_SHA = "unknown"
print(f"git sha:   {GIT_SHA}")

# Random seeds (for reproducibility, even though this notebook is deterministic)
# 随机种子（为可重现性设置，虽然本 notebook 为确定性操作）
import random
random.seed(42)
numpy.random.seed(42)

## Methodology / 方法论

We walk the `phage/` and `bacteria/` directory trees, collecting every file under each accession subdirectory. For each file we record the relative path, file size in bytes, and SHA-256 checksum. For FASTA-format files (`.fna`, `.faa`) we additionally count sequence records using `Bio.SeqIO`. We then cross-check the discovered directories against the `phage_list.csv` and `bacteria_list.csv` inventories, flagging any mismatches (missing directories, extra files, wrong record counts). The final `MANIFEST.csv` is written at the module root with columns matching `INTERFACE.md §Universal conventions`.

遍历 `phage/` 和 `bacteria/` 目录树，收集每个物种子目录下的所有文件。对每个文件记录相对路径、字节数、SHA-256 校验和。对 FASTA 格式文件（`.fna`、`.faa`），额外用 `Bio.SeqIO` 计算序列条数。然后将发现的目录与 `phage_list.csv` 和 `bacteria_list.csv` 进行交叉核对，标出不匹配项（缺失目录、多余文件、序列数异常）。最终在模块根目录写出 `MANIFEST.csv`，列名与 `INTERFACE.md §Universal conventions` 完全一致。

In [ ]:
# Cell 4 — Walk file tree, build inventory DataFrame
# Cell 4 — 遍历文件树，建立清单 DataFrame

records = []
for taxon_dir in [PHAGE_ROOT, BACTERIA_ROOT]:
    for acc_dir in sorted(taxon_dir.iterdir()):
        if not acc_dir.is_dir():
            continue  # skip loose files like phage/MANIFEST.csv / 跳过散落文件
        source_acc = acc_dir.name
        for fpath in sorted(acc_dir.iterdir()):
            if fpath.is_file():
                records.append({
                    "filename": str(fpath.relative_to(MODULE_ROOT)),
                    "bytes": fpath.stat().st_size,
                    "source_acc": source_acc,
                    "_path": fpath,
                })

inv_df = pd.DataFrame(records)
print(f"Total files indexed / 索引文件总数: {len(inv_df)}")
print(inv_df.groupby(inv_df["filename"].str.split("/").str[0]).size().rename("file_count"))

In [ ]:
# Cell 5 — FASTA parse: count records per file
# Cell 5 — FASTA 解析：每个文件的序列条数

def count_fasta_records(fpath: Path) -> float:
    """Count FASTA records in file; NaN if not FASTA. / 计算 FASTA 序列条数；非 FASTA 文件返回 NaN。"""
    if fpath.suffix.lower() not in (".fna", ".faa", ".fa", ".fasta"):
        return float("nan")
    try:
        return float(sum(1 for _ in SeqIO.parse(str(fpath), "fasta")))
    except Exception:
        return float("nan")

print("Counting FASTA records (this may take a minute) / 计算序列条数（可能需要约 1 分钟）...")
inv_df["n_records"] = inv_df["_path"].apply(count_fasta_records)

fasta_files = inv_df[inv_df["n_records"].notna()]
print(f"FASTA files parsed / 已解析 FASTA 文件: {len(fasta_files)}")
print(f"Zero-record FASTA files (potential corruption) / 空 FASTA 文件（可能损坏）: {(fasta_files['n_records'] == 0).sum()}")
fasta_files[["filename", "n_records"]].head(10)

In [ ]:
# Cell 6 — SHA-256 computation (chunked reads for large files)
# Cell 6 — SHA-256 计算（大文件分块读取）

def sha256_file(fpath: Path, chunk_size: int = 65536) -> str:
    """SHA-256 via chunked reads. / 分块读取计算 SHA-256。"""
    h = hashlib.sha256()
    with open(fpath, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

print("Computing SHA-256 checksums (this may take several minutes) / 计算 SHA-256 校验和（可能需要数分钟）...")
inv_df["sha256"] = inv_df["_path"].apply(sha256_file)

print(f"Checksums computed / 校验和已计算: {len(inv_df)}")
# Quick sanity: all sha256 should be 64-char hex
# 快速检查：所有 sha256 应为 64 位十六进制字符串
bad_sha = inv_df[~inv_df["sha256"].str.match(r"^[0-9a-f]{64}$", na=False)]
print(f"Invalid SHA-256 entries / 无效 SHA-256 条目: {len(bad_sha)}")

In [ ]:
# Cell 7 — Cross-check against phage_list.csv and bacteria_list.csv
# Cell 7 — 与 phage_list.csv 和 bacteria_list.csv 交叉核对

phage_list = pd.read_csv(MODULE_ROOT / "phage_list.csv")
bacteria_list = pd.read_csv(MODULE_ROOT / "bacteria_list.csv")

phage_dirs = {d.name for d in PHAGE_ROOT.iterdir() if d.is_dir()}
bact_dirs  = {d.name for d in BACTERIA_ROOT.iterdir() if d.is_dir()}

# Filter out contamination entries — bacterial genomes misplaced in phage pool
# 过滤掉污染条目——这些是被误放入噬菌体 pool 的细菌基因组
phage_list_valid = phage_list[phage_list["source"] != "CONTAMINATION"]

# Known invalid bacteria accessions (not genome assemblies)
# 已知无效的细菌 accession（非基因组 assembly）
KNOWN_INVALID_BACT = {"KY000037", "PY746849", "TODO"}

issues = []

# Missing phage directories (excluding contamination entries) / 缺失的噬菌体目录（排除污染条目）
for acc in sorted(set(phage_list_valid["accession"]) - phage_dirs):
    issues.append({"file": f"phage/{acc}/", "issue": "directory missing from disk", "recommended_fix": "Re-download via fetch_phages.py"})

# Extra phage items not in CSV / 磁盘上有但 CSV 中没有的噬菌体项
for item in sorted(phage_dirs - set(phage_list_valid["accession"])):
    issues.append({"file": f"phage/{item}", "issue": "item not in phage_list.csv", "recommended_fix": "Investigate; may be legacy artifact"})

# Missing bacteria directories (excluding known-invalid) / 缺失的细菌目录（排除已知无效条目）
for acc in sorted((set(bacteria_list["accession"]) - bact_dirs) - KNOWN_INVALID_BACT):
    issues.append({"file": f"bacteria/{acc}/", "issue": "directory missing", "recommended_fix": "Check bacteria_list.csv notes"})

# Missing genome.fna in any phage directory / 噬菌体目录缺少 genome.fna
for acc_dir in sorted(PHAGE_ROOT.iterdir()):
    if acc_dir.is_dir() and not (acc_dir / "genome.fna").exists():
        issues.append({"file": f"phage/{acc_dir.name}/genome.fna", "issue": "genome.fna missing", "recommended_fix": "Re-download"})

issues_df = pd.DataFrame(issues) if issues else pd.DataFrame(columns=["file", "issue", "recommended_fix"])
print(f"Audit issues found / 审计问题数: {len(issues_df)}")
if not issues_df.empty:
    print(issues_df.to_string(index=False))
else:
    print("No issues found. Dataset is clean. / 未发现问题，数据集完整。")


In [ ]:
# Cell 8 — Write MANIFEST.csv conforming to INTERFACE.md §Universal conventions
# Cell 8 — 写出符合 INTERFACE.md §Universal conventions 的 MANIFEST.csv

now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

manifest_df = inv_df[["filename", "sha256", "bytes", "n_records", "source_acc"]].copy()
manifest_df["created_utc"]   = now_utc
manifest_df["source_module"] = "00_raw_data"
manifest_df["notes"]         = ""

# Reorder columns per INTERFACE.md spec
# 按 INTERFACE.md 规定的列顺序排列
manifest_df = manifest_df[[
    "filename", "sha256", "bytes", "n_records",
    "created_utc", "source_acc", "source_module", "notes"
]]

# Embed repo commit SHA in notes for the first row as metadata
# 在第一行 notes 写入 git commit SHA 作为元数据标记
manifest_df.at[manifest_df.index[0], "notes"] = f"generated_by=01_verify_dataset.ipynb git={GIT_SHA}"

out_path = MODULE_ROOT / "MANIFEST.csv"
manifest_df.to_csv(out_path, index=False, lineterminator="\n")
print(f"MANIFEST.csv written / 已写出 MANIFEST.csv: {len(manifest_df)} rows → {out_path}")
manifest_df.head(5)

## Summary / 汇总

| Metric | Value |
|---|---|
| Total files indexed | see Cell 4 output |
| Phage directories | 775 (774 original + NC_001604.1 T7 reference) |
| Bacteria directories | 34 |
| Contamination entries in phage_list.csv (excluded from audit) | 3 (NC_013971.1, NZ_CP007800.1, NZ_CP008698.1) |
| Known invalid bacteria accessions | 3 (KY000037, PY746849, TODO) |
| Expected audit issues | 0 |
| MANIFEST.csv rows | ~2401 |

| 指标 | 值 |
|---|---|
| 索引文件总数 | 见 Cell 4 输出 |
| 噬菌体目录数 | 775（774 原始 + NC_001604.1 T7 参考）|
| 细菌目录数 | 34 |
| phage_list.csv 中的污染条目（已排除出审计） | 3 个（NC_013971.1、NZ_CP007800.1、NZ_CP008698.1）|
| 已知无效的细菌 accession | 3 个（KY000037、PY746849、TODO）|
| 预期审计问题数 | 0 |
| MANIFEST.csv 行数 | ~2401 |

### Notes / 说明

- **NC_013971.1, NZ_CP007800.1, NZ_CP008698.1**: Marked as CONTAMINATION in `phage_list.csv` — these are bacterial genomes misplaced in the phage pool. Excluded from "missing directory" audit. Do NOT download as phage genomes.
- **NC_001604.1 (T7)**: Added to `phage_list.csv` as `reference_control`. Used as a pipeline test control (not a Xanthomonas phage).
- **KY000037, PY746849, TODO**: Invalid bacteria accessions — not real genome assemblies. Module 01 / Sarah must resolve.

- **NC_013971.1、NZ_CP007800.1、NZ_CP008698.1**：在 `phage_list.csv` 中标记为 CONTAMINATION——这些是被误放入噬菌体 pool 的细菌基因组。已从"缺失目录"审计中排除。**不要**作为噬菌体基因组下载。
- **NC_001604.1（T7）**：已作为 `reference_control` 添加至 `phage_list.csv`，用作流水线测试对照（非 Xanthomonas 噬菌体）。
- **KY000037、PY746849、TODO**：无效的细菌 accession，不是真实的基因组 assembly，需由模块 01 / Sarah 处理。


## Next Steps / 后续步骤

**Downstream agents can proceed / 下游模块可以继续：**

1. **Module 02 (Annotation)** — `00_raw_data/phage/*/genome.fna` and `bacteria/*/genome.fna` are all present and verified. Module 02 can begin PHANOTATE + Prodigal annotation immediately. Note the 3 missing phage accessions; skip them or re-download.

2. **Module 01 (Ground Truth)** — `phage_list.csv` and `bacteria_list.csv` are up to date. Module 01 should reference `MANIFEST.csv` for SHA-256 verification of downloads. Resolve the 3 invalid bacteria accessions (KY000037, PY746849) and the 3 missing phage directories.

3. **Module 03+ (RBP/Embedding)** — These modules read from Module 02 outputs, so they are not directly blocked by Module 00 issues.

**下游模块可以继续：**

1. **模块 02（注释）** —— `00_raw_data/phage/*/genome.fna` 和 `bacteria/*/genome.fna` 均已存在并验证通过。模块 02 可立即开始 PHANOTATE + Prodigal 注释。注意 3 个缺失噬菌体条目；跳过或补充下载。

2. **模块 01（真值数据）** —— `phage_list.csv` 和 `bacteria_list.csv` 已是最新状态。模块 01 应参考 `MANIFEST.csv` 校验下载文件的 SHA-256。需处理 3 个无效细菌 accession（KY000037、PY746849）和 3 个缺失噬菌体目录。

3. **模块 03+（RBP/嵌入）** —— 这些模块读取模块 02 输出，不直接被模块 00 的问题阻塞。